# Northwind Dashboard – Development

This notebook is the **dashboard preparation/prototyping layer**.

It does not depend on the memory/state of the previous notebook.  
The analysis notebook exports the final DataFrames to `Northwind_Dashboard_Development/`, and this notebook loads them.

**Final interactive dashboard:** `dashboard/app.py` using Streamlit + Plotly.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

PROJECT_PATH = Path(r"C:\Users\User\Desktop\ספיר\אישי\קורס דאטה אנליסט אנליזה\Northwind project\Northwind Profitability & Growth Analysis using Python")
DATA_PATH = PROJECT_PATH / "Northwind_Dashboard_Development"

print("Northwind_Dashboard_Development:", DATA_PATH)


## 1. Load the outputs created by the analysis notebook

The dashboard should read **prepared analytical data**, not repeat the entire cleaning and profitability analysis.


In [ ]:
files = {
    "profitability": "profitability.csv",
    "product_profitability": "product_profitability.csv",
    "category_profitability": "category_profitability.csv",
    "customer_profitability": "customer_profitability.csv",
    "volume_margin": "volume_margin.csv",
    "employee_profitability": "employee_profitability.csv",
    "country_profitability": "country_profitability.csv",
    "monthly_profitability": "monthly_profitability.csv",
    "discount_analysis": "discount_analysis.csv",
    "monthly_active_customers": "monthly_active_customers.csv",
}

data = {Northwind_Dashboard_Development: None for Northwind_Dashboard_Development in files.keys()}

for Northwind_Dashboard_Development, filename in files.items():
    path = DATA_PATH / filename
    data[Northwind_Dashboard_Development] = pd.read_csv(path)
    print(f"{Northwind_Dashboard_Development}: {data[Northwind_Dashboard_Development].shape}")

profitability = data["profitability"]
product_profitability = data["product_profitability"]
category_profitability = data["category_profitability"]
customer_profitability = data["customer_profitability"]
volume_margin = data["volume_margin"]
employee_profitability = data["employee_profitability"]
country_profitability = data["country_profitability"]
monthly_profitability = data["monthly_profitability"]
discount_analysis = data["discount_analysis"]
monthly_active_customers = data["monthly_active_customers"]

profitability["OrderDate"] = pd.to_datetime(profitability["OrderDate"])
monthly_profitability["Month"] = pd.to_datetime(monthly_profitability["Month"])
monthly_active_customers["Month"] = pd.to_datetime(monthly_active_customers["Month"])


## 2. Executive KPI calculations

In [ ]:
total_revenue = profitability["Revenue"].sum()
total_cogs = profitability["COGS"].sum()
total_profit = profitability["GrossProfit"].sum()
gross_margin = total_profit / total_revenue if total_revenue else np.nan
total_orders = profitability["OrderID"].nunique()
total_customers = profitability["CustomerID"].nunique()

aov = total_revenue / total_orders if total_orders else np.nan
profit_per_customer = total_profit / total_customers if total_customers else np.nan
revenue_per_customer = total_revenue / total_customers if total_customers else np.nan

orders_per_customer = profitability.groupby("CustomerID")["OrderID"].nunique()
repeat_purchase_rate = (orders_per_customer.gt(1).mean() * 100)

customer_revenue = profitability.groupby("CustomerID")["Revenue"].sum().sort_values(ascending=False)
customer_profit = profitability.groupby("CustomerID")["GrossProfit"].sum().sort_values(ascending=False)

top10_revenue_concentration = customer_revenue.head(10).sum() / total_revenue * 100
top10_profit_concentration = customer_profit.head(10).sum() / total_profit * 100

kpis = pd.DataFrame({
    "KPI": [
        "Total Revenue", "Gross Profit", "Gross Margin", "Orders",
        "Customers", "Average Order Value", "Revenue per Customer",
        "Profit per Customer", "Repeat Purchase Rate",
        "Top 10 Revenue Concentration", "Top 10 Profit Concentration"
    ],
    "Value": [
        total_revenue, total_profit, gross_margin, total_orders,
        total_customers, aov, revenue_per_customer,
        profit_per_customer, repeat_purchase_rate / 100,
        top10_revenue_concentration / 100, top10_profit_concentration / 100
    ]
})

kpis


## 3. Filters

In [ ]:
years = sorted(profitability["OrderDate"].dt.year.dropna().unique())
categories = sorted(profitability["CategoryName"].dropna().unique()) if "CategoryName" in profitability.columns else []

selected_year = years[-1] if years else None
selected_category = "All"

filtered = profitability.copy()

if selected_year is not None:
    filtered = filtered[filtered["OrderDate"].dt.year == selected_year]

if selected_category != "All" and "CategoryName" in filtered.columns:
    filtered = filtered[filtered["CategoryName"] == selected_category]

filtered.shape


## 4. Example dashboard charts

In [ ]:
# Revenue and Gross Profit over time
monthly = (
    filtered.assign(Month=filtered["OrderDate"].dt.to_period("M").dt.to_timestamp())
    .groupby("Month", as_index=False)
    .agg(Revenue=("Revenue", "sum"),
         GrossProfit=("GrossProfit", "sum"))
)

fig = px.line(
    monthly,
    x="Month",
    y=["Revenue", "GrossProfit"],
    markers=True,
    title="Revenue & Gross Profit Over Time"
)
fig.show()


In [ ]:
# Top 10 products by Gross Profit
top_products = (
    filtered.groupby(["ProductID", "ProductName"], as_index=False)
    .agg(GrossProfit=("GrossProfit", "sum"),
         Revenue=("Revenue", "sum"),
         UnitsSold=("Quantity", "sum"))
    .sort_values("GrossProfit", ascending=False)
    .head(10)
)

fig = px.bar(
    top_products.sort_values("GrossProfit"),
    x="GrossProfit",
    y="ProductName",
    orientation="h",
    title="Top 10 Products by Gross Profit"
)
fig.show()


In [ ]:
# Revenue vs Gross Margin
product_view = (
    filtered.groupby(["ProductID", "ProductName"], as_index=False)
    .agg(Revenue=("Revenue", "sum"),
         GrossProfit=("GrossProfit", "sum"),
         UnitsSold=("Quantity", "sum"))
)
product_view["GrossMargin"] = np.where(
    product_view["Revenue"] != 0,
    product_view["GrossProfit"] / product_view["Revenue"],
    np.nan
)

fig = px.scatter(
    product_view,
    x="Revenue",
    y="GrossMargin",
    size="UnitsSold",
    hover_name="ProductName",
    title="Revenue vs Gross Margin"
)
fig.update_yaxes(tickformat=".0%")
fig.show()


## 5. What this notebook is for

Use this notebook to prototype and validate:

- KPI definitions
- filters
- charts
- business questions
- insight candidates

Once the visual design is approved, the same logic is implemented in `dashboard/app.py`.
